# Exploratory Data Analysis of the Cleaned Master Dataset

This notebook explores the cleaned merged spare-parts pricing dataset used in the thesis. The goal is to understand dataset composition, dominant part types, repeated listing behavior, target-variable characteristics, and the overall structure of the available market features before any feature engineering, demand proxy construction, or modeling.

Repeated observations across scrape dates are expected in this dataset. The same `product_id` can appear multiple times because listings were observed repeatedly over time.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

## Load Dataset

A single path variable is defined below so the notebook is easy to maintain if the cleaned dataset location changes later.

In [ ]:
data_path = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/cleaned/clean_master_dataset.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path, skipinitialspace=True)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

if "scrape_date" in df.columns:
    df["scrape_date"] = pd.to_datetime(df["scrape_date"], errors="coerce")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print()
print("Column names:")
print(df.columns.tolist())
print()
print("Dtype summary:")
print(df.dtypes.value_counts())

display(df.head())


## Dataset Overview

This section establishes the overall size and structure of the dataset, with particular attention to repeated listing observations and missing values.

In [ ]:
overview = {
    "rows": len(df),
    "columns": len(df.columns),
    "unique_product_id": df["product_id"].nunique(),
    "unique_scrape_dates": df["scrape_date"].nunique(),
}
overview["avg_observations_per_product_id"] = overview["rows"] / overview["unique_product_id"]

overview_table = pd.DataFrame.from_dict(overview, orient="index", columns=["value"])
display(overview_table)

missing_overview = (
    pd.DataFrame(
        {
            "column_name": df.columns,
            "missing_count": [int(df[column].isna().sum()) for column in df.columns],
            "missing_percentage": [df[column].isna().mean() * 100 for column in df.columns],
        }
    )
    .sort_values(["missing_percentage", "missing_count"], ascending=[False, False])
    .reset_index(drop=True)
)

print("Missing values overview:")
display(missing_overview.head(15))

## Target Variable Analysis (`price`)

The target variable is `price`. The plots below compare the raw price distribution with a log-transformed version in order to assess skewness and the influence of high-value observations.

In [ ]:
price_summary = df["price"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("price")
display(price_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["price"], bins=40, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Price Distribution")
axes[0].set_xlabel("Price")

sns.histplot(np.log1p(df["price"]), bins=40, kde=True, ax=axes[1], color="#F58518")
axes[1].set_title("log1p(Price) Distribution")
axes[1].set_xlabel("log1p(price)")

sns.boxplot(x=df["price"], ax=axes[2], color="#54A24B")
axes[2].set_title("Price Boxplot")
axes[2].set_xlabel("Price")

plt.tight_layout()
plt.show()

The raw price distribution is expected to be right-skewed because a relatively small number of expensive parts can stretch the upper tail. The log-transformed view helps assess whether later modeling may benefit from scale compression.

## Listing Structure and Repeated Observations

Repeated observations are a defining feature of this dataset. The same `product_id` may be observed across several scrape dates, so repetition should be interpreted as part of the panel-like listing structure rather than treated as an ordinary duplicate problem.

In [ ]:
observations_per_listing = df.groupby("product_id").size().rename("observation_count")
observation_count_distribution = observations_per_listing.value_counts().sort_index().rename_axis("observations_per_product_id").reset_index(name="listing_count")

display(observations_per_listing.describe().to_frame("observation_count"))
display(observation_count_distribution)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(observations_per_listing, bins=range(1, observations_per_listing.max() + 2), discrete=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Observations per Product ID")
axes[0].set_xlabel("Number of observations")
axes[0].set_ylabel("Count of product IDs")

sns.barplot(data=observation_count_distribution, x="observations_per_product_id", y="listing_count", ax=axes[1], color="#F58518")
axes[1].set_title("How Often Listings Reappear")
axes[1].set_xlabel("Observations per product_id")
axes[1].set_ylabel("Number of listings")

plt.tight_layout()
plt.show()

## Part and Category Analysis

This section focuses on the composition of the spare-parts dataset. It is especially important for understanding which part groups dominate the sample and whether the target variable behaves differently across categories.

The `subcategory` field should be interpreted carefully because it mixes subtype labels with positional descriptors such as `left`, `right`, and `rear`.

In [ ]:
TOP_N = 15

category_counts = df["category"].value_counts().head(TOP_N).rename_axis("category").reset_index(name="row_count")
subcategory_counts = df["subcategory"].value_counts().head(TOP_N).rename_axis("subcategory").reset_index(name="row_count")
part_name_counts = df["part_name"].value_counts().head(TOP_N).rename_axis("part_name").reset_index(name="row_count")

category_listing_summary = (
    df.groupby("category")
    .agg(
        row_count=("product_id", "size"),
        unique_product_id=("product_id", "nunique"),
        median_price=("price", "median"),
        mean_price=("price", "mean"),
    )
    .sort_values("row_count", ascending=False)
)

print("Top categories by row count:")
display(category_counts)
print("Top subcategories by row count:")
display(subcategory_counts)
print("Top part names by row count:")
display(part_name_counts)
print("Category summary:")
display(category_listing_summary.head(TOP_N))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sns.barplot(data=category_counts, y="category", x="row_count", ax=axes[0], color="#4C78A8")
axes[0].set_title("Top Categories by Count")
axes[0].set_xlabel("Number of rows")
axes[0].set_ylabel("")

sns.barplot(data=part_name_counts, y="part_name", x="row_count", ax=axes[1], color="#F58518")
axes[1].set_title("Top Part Names by Count")
axes[1].set_xlabel("Number of rows")
axes[1].set_ylabel("")

category_price_plot = category_listing_summary.head(10).sort_values("median_price", ascending=False).reset_index()
sns.barplot(data=category_price_plot, y="category", x="median_price", ax=axes[2], color="#54A24B")
axes[2].set_title("Median Price by Category")
axes[2].set_xlabel("Median price")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

## Brand and Model Analysis

Because the dataset combines spare-parts listings with market context, it is useful to examine how heavily the data is concentrated in a small number of brands and models.

In [ ]:
TOP_N_BRAND = 10

brand_summary = (
    df.groupby("brand")
    .agg(
        row_count=("product_id", "size"),
        unique_product_id=("product_id", "nunique"),
        median_price=("price", "median"),
    )
    .sort_values("row_count", ascending=False)
)

model_summary = (
    df.groupby("model")
    .agg(
        row_count=("product_id", "size"),
        unique_product_id=("product_id", "nunique"),
        median_price=("price", "median"),
    )
    .sort_values("row_count", ascending=False)
)

print("Brand summary:")
display(brand_summary.head(TOP_N_BRAND))
print("Model summary:")
display(model_summary.head(TOP_N_BRAND))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

brand_plot = brand_summary.head(TOP_N_BRAND).reset_index()
sns.barplot(data=brand_plot, x="brand", y="row_count", ax=axes[0], color="#4C78A8")
axes[0].set_title("Top Brands by Count")
axes[0].set_xlabel("Brand")
axes[0].set_ylabel("Number of rows")
axes[0].tick_params(axis="x", rotation=30)

model_plot = model_summary.head(TOP_N_BRAND).reset_index()
sns.barplot(data=model_plot, x="model", y="median_price", ax=axes[1], color="#F58518")
axes[1].set_title("Median Price by Model")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("Median price")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Mileage and Year-Related Analysis

Mileage and model-year fields are likely to be important later in the project. The goal here is not to engineer new features yet, but to understand their basic ranges, missingness, and shape.

Very low mileage values were already handled in the cleaning stage where needed.

In [ ]:
mileage_summary = df["mileage"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("mileage")
display(mileage_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df["mileage"].dropna(), bins=40, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Mileage Distribution")
axes[0].set_xlabel("Mileage")

sns.boxplot(x=df["mileage"], ax=axes[1], color="#F58518")
axes[1].set_title("Mileage Boxplot")
axes[1].set_xlabel("Mileage")

plt.tight_layout()
plt.show()

In [ ]:
year_columns = ["year_start", "year_end", "year_span", "year_mid"]
year_summary = df[year_columns].describe().T
display(year_summary)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for axis, column in zip(axes, year_columns):
    sns.histplot(df[column], bins=30, kde=False, ax=axis, color="#54A24B")
    axis.set_title(f"Distribution of {column}")
    axis.set_xlabel(column)

plt.tight_layout()
plt.show()

These plots help identify whether the sample is concentrated in specific production-year bands and whether the year-span variables carry variation that may later matter for price formation.

## Scrape Date and Temporal Overview

The cleaned dataset spans a short repeated scraping window. This section checks whether row counts and price summaries vary meaningfully across scrape dates.

In [ ]:
scrape_date_summary = (
    df.groupby("scrape_date")
    .agg(
        row_count=("product_id", "size"),
        unique_product_id=("product_id", "nunique"),
        mean_price=("price", "mean"),
        median_price=("price", "median"),
    )
    .reset_index()
    .sort_values("scrape_date")
)

display(scrape_date_summary)

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=scrape_date_summary, x="scrape_date", y="row_count", marker="o", ax=ax, color="#4C78A8")
ax.set_title("Listing Counts by Scrape Date")
ax.set_xlabel("Scrape date")
ax.set_ylabel("Number of rows")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Numeric Feature Overview

A concise numeric summary is useful for checking scale differences across the main modeling candidates. The correlation view is intentionally restricted to a small set of interpretable variables so that it remains readable.

In [ ]:
main_numeric_columns = [
    "price",
    "mileage",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "model_total_registered",
    "model_median_vehicle_age",
    "model_median_mileage",
    "model_share_of_market",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_median_mileage",
    "brand_share_of_market",
]

numeric_overview = df[main_numeric_columns].describe().T[["count", "mean", "std", "min", "50%", "max"]]
display(numeric_overview)

In [ ]:
correlation_columns = [
    "price",
    "mileage",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "model_total_registered",
    "model_median_vehicle_age",
    "model_median_mileage",
    "model_share_of_market",
    "brand_total_registered",
    "brand_median_vehicle_age",
]

correlation_matrix = df[correlation_columns].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=False)
plt.title("Correlation Matrix for Selected Numeric Variables")
plt.tight_layout()
plt.show()

## Key Findings Summary

The final cell generates a compact summary of the most important descriptive findings for later feature engineering and demand proxy design.

In [ ]:
category_mode = df["category"].mode().iat[0]
most_common_observation_count = observations_per_listing.mode().iat[0]
most_missing_column = missing_overview.loc[0, "column_name"]
most_missing_percentage = missing_overview.loc[0, "missing_percentage"]

summary_lines = [
    "### Main Findings",
    f"- The cleaned master dataset contains **{len(df):,} rows** and **{len(df.columns):,} columns**.",
    f"- There are **{df['product_id'].nunique():,} unique product IDs** observed across **{df['scrape_date'].nunique():,} scrape dates**, which confirms a repeated-observation dataset structure rather than a simple one-row-per-listing dataset.",
    f"- The average number of observations per `product_id` is **{len(df) / df['product_id'].nunique():.2f}**, and the most common listing observation count is **{most_common_observation_count}**.",
    f"- The price distribution is right-skewed, so extreme values should be kept in mind when selecting transformations or robust summary measures later.",
    f"- The largest part category in the dataset is **{category_mode}**, which indicates strong category concentration in the sample.",
    f"- The column with the highest missingness is **{most_missing_column}** at **{most_missing_percentage:.2f}%**, which is relevant for later feature preparation.",
    f"- The short scrape window means temporal variation should be interpreted as repeated market observation over a limited period rather than a long-run time series.",
    f"- These patterns suggest that later feature engineering should respect repeated listing structure, skewed prices, concentrated part groups, and non-trivial mileage missingness.",
]

display(Markdown("\n".join(summary_lines)))
